# **Riconoscere automaticamente il livello CEFR di un testo: un confronto tra metodi a regole, apprendimento automatico e LLM**
Il progetto ha come obiettivo il classificare automaticamente un testo in base al livello linguistico, facendo riferimento al Quadro Comune Europeo di Riferimento per le lingue (*QCER*), in inglese *CEFR* (Common European Framework of Reference for languages). Verranno esplorati vari metodi, raggruppati in 3 approcci differenti (*Rule-Based, Machine Learning* e *LLM*), e ne verrà valutata la performance su un dataset di testing.



## **Costruzione dei modelli, training e primo testing**
**NoteBook 2/3.**


In questo Notebook vengono caricati i vari dataset preprocessati e vengono costruiti i vari modelli. Come ultimo step, vengono poi creati due dizionari contenenti i *classification report* per ogni modello.

##Installazione / Importazione librerie

In [ ]:
!pip install fasttext syllables

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 22.8 MB/s eta 0:00:00
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4647417 sha256=1b808b3c3b8479e47d0172c8d8f344fc2e03bde96c52042bcd5c3ee8aebe9070
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 8.7.1
    Uninstalling importlib_metadata-8.7.1:
      Successfully uninstalled importlib_metadata-8.7.1


In [ ]:
import json
import math
import os
import random
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import fasttext
import gdown
import nltk
import requests
import spacy
import syllables

from google import genai

from nltk.tokenize import word_tokenize, sent_tokenize

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

nltk.download("punkt_tab")
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


##**Importazione materiali**
La cartella con i materiali (testi, vettori, bow e altro) vengono scaricati...

In [ ]:
url = r'https://drive.google.com/file/d/1-A14qvPrDNIqNYeXeFVYdAaJXf82Ny6S/view?usp=sharing'
output_path = 'material_notebook_2.zip'
gdown.download(url, output_path, quiet=False,fuzzy=True)

Downloading...
From (original): https://drive.google.com/uc?id=1-A14qvPrDNIqNYeXeFVYdAaJXf82Ny6S
From (redirected): https://drive.google.com/uc?id=1-A14qvPrDNIqNYeXeFVYdAaJXf82Ny6S&confirm=t&uuid=19ff9734-c90c-4404-9814-a9ce12ba08eb
To: /content/material_notebook_2.zip
100%|██████████| 586M/586M [00:08<00:00, 66.9MB/s]


'material_notebook_2.zip'

... e estratti:

In [ ]:
zip_file_path = r'/content/material_notebook_2.zip'
extract_dir = r'/content/material_notebook_2'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

Viene creato un dizionario "texts" contenente tutti i testi:

In [ ]:
texts = {
    "main_dataset":{
        "0.raw":{
            "txt":[],
            "train":[],
            "test":[]
        },
        "1.lowercased":{
            "txt":[],
            "train":[],
            "test":[]
        },
        "2.lowercased_lemmatized":{
            "txt":[],
            "train":[],
            "test":[]
        },
        "3.lowercased_nostopwords":{
            "txt":[],
            "train":[],
            "test":[]
        },
        "4.lowercased_nostopwords_lemmatized":{
            "txt":[],
            "train":[],
            "test":[]
        },
        "5.lowercased_nopunctuation":{
            "txt":[],
            "train":[],
            "test":[]
        },
        "6.lowercased_nopunctuation_lemmatized":{
            "txt":[],
            "train":[],
            "test":[]
        },
        "7.lowercased_nopunctuation_nostopwords":{
            "txt":[],
            "train":[],
            "test":[]
        },
        "8.lowercased_nopunctuation_nostopwords_lemmatized":{
            "txt":[],
            "train":[],
            "test":[]
        }
    },
    "generalized_dataset":{
        "0.raw":{
            "test":[]
        },
        "1.lowercased":{
            "test":[]
        },
        "2.lowercased_lemmatized":{
            "test":[]
        },
        "3.lowercased_nostopwords":{
            "test":[]
        },
        "4.lowercased_nostopwords_lemmatized":{
            "test":[]
        },
        "5.lowercased_nopunctuation":{
            "test":[]
        },
        "6.lowercased_nopunctuation_lemmatized":{
            "test":[]
        },
        "7.lowercased_nopunctuation_nostopwords":{
            "test":[]
        },
        "8.lowercased_nopunctuation_nostopwords_lemmatized":{
            "test":[]
        }
    }
}

E i testi vengono importati e inseriti nel dizionario. Viene inoltre stampata una "struttura" per vedere come sono stati divisi i testi e se sono stati tutti inclusi:

In [ ]:
for folder in [f for f in os.listdir("/content/material_notebook_2/preprocessed") if "dataset" in f]:
  print(folder)
  for subfolder in [f for f in os.listdir(f"/content/material_notebook_2/preprocessed/{folder}") if "." in f]:
    print("     ",subfolder)
    for file in [f for f in os.listdir(f"/content/material_notebook_2/preprocessed/{folder}/{subfolder}") if ".txt" in f or "train" in f or "test" in f]:
      with open(f"/content/material_notebook_2/preprocessed/{folder}/{subfolder}/{file}", "r") as f:
        content = f.readlines()
        texts[folder][subfolder][file.split(".")[1]] = [item.strip() for item in content]
        print(f"          {file}:   {len(content)} testi")

main_dataset
      5.lowercased_nopunctuation
          main_dataset.test:   9990 testi
          main_dataset.txt:   49950 testi
          main_dataset.train:   39960 testi
      7.lowercased_nopunctuation_nostopwords
          main_dataset.test:   9990 testi
          main_dataset.txt:   49950 testi
          main_dataset.train:   39960 testi
      3.lowercased_nostopwords
          main_dataset.test:   9990 testi
          main_dataset.txt:   49950 testi
          main_dataset.train:   39960 testi
      4.lowercased_nostopwords_lemmatized
          main_dataset.test:   9990 testi
          main_dataset.txt:   49950 testi
          main_dataset.train:   39960 testi
      2.lowercased_lemmatized
          main_dataset.test:   9990 testi
          main_dataset.txt:   49950 testi
          main_dataset.train:   39960 testi
      8.lowercased_nopunctuation_nostopwords_lemmatized
          main_dataset.test:   9990 testi
          main_dataset.txt:   49950 testi
          main_dataset.tra

###**Importazioni varie**

In [ ]:
with open(r'/content/material_notebook_2/other_stuff/connectors.txt','r',encoding='utf-8') as conn_file:
  connectors = set(conn_file.read().splitlines())

##**Rule-Based**
I primi metodi sono "a regole": utilizzano infatti delle regole precise (e delle formule) per determinare il livello linguistico di un testo. Si dividono in due gruppi:
* Analisi lessicale;
* Formule di leggibilià.

###**Analisi lessicale**
Un primo approccio consiste nell"utilizzare il "valore linguistico" delle parole, lasciando da parte la sintassi e qualsiasi altro elemento sovra-lessicale. Ogni testo è composto da determinate parole e, secondo questo approccio, è probabile che ogni livello linguistico presenti un lessico determinato e discriminante.

Possono essere testati due metodi:
1. "*Raw Counts*": conteggio di parole per ogni livello linguistico;
2. *Weight Extraction*: estrazione di pesi di ogni parola per ogni livello linguistico.

Prima di tutto, vengono importate due strutture dati:
1. *labelled_words* : dizionario di parole accoppiate al loro livello linguistico ;
2. *word_filter* : lista di parole inglesi, utilizzata nel secondo metodo per filtrare le parole ottenute dalla tokenizzazione dei vari testi.

In [ ]:
with open("/content/material_notebook_2/other_stuff/labelled_words.json", "r", encoding='utf-8') as f:
  labelled_words = json.load(f)

In [ ]:
word_filter = set([word.strip().lower() for word in requests.get(r"https://raw.githubusercontent.com/dwyl/english-words/refs/heads/master/words.txt").text.splitlines()])

####**Raw counts**
Dato ogni parola ha un livello CEFR di riferimento, una strategia può trovarsi nell"usare la quantità di parole di ogni livello per predire il testo.


Viene definita una prima funzione che ha lo scopo di ritornare un dizionario *counter*, dopo aver preso come argomento un testo. In questo dizionario saranno conteggiate il numero di parole A1, A2, B1, B2 e C1.

In [ ]:
def word_level_counter(text):
  global labelled_words
  counter = {
      "__label__A1":0,
      "__label__A2":0,
      "__label__B1":0,
      "__label__B2":0,
      "__label__C1":0
            }
  tokens = word_tokenize(text)
  for token in tokens:
    if token in labelled_words and labelled_words[token] != "C2":
      counter[f"__label__{labelled_words[token]}"]+=1
  return counter

A partire dal dizionario counter ottenuto da *word_level_counter(text)*, la funzione restituisce la chiave con il valore più alto.
Quindi, in una situazione come:

    {"A1": 10, "A2": 8, "B1": 4, "B2": 2, "C1": 1, "C2": 0}

il testo sarà classificato come *__label__A1*.


In [ ]:
def raw_counts_prediction(text, return_counts=False):
  counts = word_level_counter(text)
  if return_counts:
    return max(counts, key=counts.get), counts
  else:
    return max(counts, key=counts.get)

Eseguiamo un test sui primi 10 testi della split di testing "raw"

In [ ]:
for text in texts["main_dataset"]["0.raw"]["test"][:10]:
  print(f"Predizione: {raw_counts_prediction(text)} ===> '{text[:50]}'")

Predizione: __label__A1 ===> '__label__C1 Dear Bill, Im looking forward to meeti'
Predizione: __label__A1 ===> '__label__A2 In Brasilia, this weekend, there is go'
Predizione: __label__A1 ===> '__label__B2 Dear Mr. Harry Martin, Thank you for p'
Predizione: __label__A1 ===> '__label__A1 I'm Alan Huang,I come from China.I lik'
Predizione: __label__A1 ===> '__label__A1 Hi! My name is Adriana. I am from Braz'
Predizione: __label__A1 ===> '__label__C1 i'm currently working for a global com'
Predizione: __label__A1 ===> '__label__A1 Most people in my country love playing'
Predizione: __label__A1 ===> '__label__B1 The food was delicious. The salade was'
Predizione: __label__A1 ===> '__label__B1 Hi, Elke! I'm flattered by your email,'
Predizione: __label__A1 ===> '__label__A2 hi mum and papa, I am going to the mus'


Già qui si nota che il "modello" tende a classificare la maggiorparte dei testi come A1.

####**Weight Extraction**
La seconda idea consiste nell'estrarre dei "pesi" a partire dal materiale di training. In breve, l"approccio consiste nel calcolare il peso di ogni singola parola in ognuno dei 5 livelli, tramite la formula:

\begin{equation}
\text{weight}(\text{level} \mid \text{word}) = \log\left[1 + \frac{p(\text{word} \mid \text{level})}{p(\text{word} \mid \text{corpus})}\right]
\end{equation}

In questo modo, ogni parola sarà un vettore di 5 pesi, ognuno di questi determinato dal rapporto tra:
* frequenza di quella parola in quel livello;
* frequenza di quella parola nell"intero corpus.

In questo modo, una parola può avere tre casi:
1. Frequente nel livello **E** frequente nel corpus (e viceversa, quindi rara in entrambi): avrà un **peso irrisorio**.
2. Frequente nel livello **MA** rara nel corpus: avrà un peso molto discriminante;
3. Rara nel livello **MA** frequente nel corpus: peso **ancora più irrisorio**.

Durante il testing, il testo viene tokenizzato e:

* Vengono contate le occorrenze delle parole: ad esempio, ipotizziamo che "the" appaia 100 volte;
* Le occorrenze di "the" (= *100* ) verranno moltiplicate per il vettore di pesi che la rappresenta:
       
      "the" = {A1 = peso_A1 * 100, A2 = peso_A2 * 100, ...}
* Vengono sommati tutti i vettori e la chiave con il valore più alto sarà l"etichetta del testo      

#####**A livello pratico**...
Per facilitare il procedimento, prima di tutto vengono create delle Bag Of Words per ogni livello e per l"intero corpus:

In [ ]:
def create_bows(training_data:list, save_to_file=True, file_name="bows"):
    """
    Creates a dictionary of BoWs used to extract weights for weighted lexical analysis.
    Arguments:
        -training_data: has to be a list of strings where each item is: "__label__A1 there you have some text...", "__label__B2 there you have some text..." and so on.
    Returns:
        -bows: a dictionary of bows, where each bows is "__label__1":{bow}, "__label__2":{bow}, ... , "full_corpus":{bow}
    """
    global word_filter

    #Defining "bows" dictionary
    bows = {
        "__label__A1": {},
        "__label__A2": {},
        "__label__B1": {},
        "__label__B2": {},
        "__label__C1": {},
        "full_corpus": {}
            }

    #Filling up the "bows" dictionary
    tot = len(training_data)
    for pair in training_data:
        if training_data.index(pair) % 1000 == 0:
          print(f"BoW n°{training_data.index(pair)} out of {tot}")
        label, text = pair.split(" ", maxsplit=1)
        for word in [w for w in word_tokenize(text) if w in word_filter]:
            if word in bows["full_corpus"]:
                bows["full_corpus"][word] += 1
            else:
                bows["full_corpus"][word] = 1

            if word in bows[label]:
                bows[label][word] += 1
            else:
                bows[label][word] = 1
    if save_to_file:
      os.makedirs('lexical_analysis/bows',exist_ok=True)
      with open(os.path.join("/content/lexical_analysis/bows", f"bows_{file_name}.json"),mode="w",encoding="utf-8") as file:
          json.dump(bows,file,indent=4)
    return bows

Viene calcolato il peso tramite la formula:

In [ ]:
def compute_weight(bows,level,word):
    """
    Computes the weight for a level and a word according to the formula:
        - weight( level | word ) = log [  1 +  p(word|level) / p(word|corpus) ]
    Arguments:
        -level: the level it computes the weight for;
        -word: the word it computes the weight for.
    """
    word_occurrences_in_level = bows[level].get(word,0)
    total_words_in_level = sum(bows[level].values())

    word_occurrences_in_corpus = bows["full_corpus"].get(word,0)
    total_words_in_corpus = sum(bows["full_corpus"].values())
    try:
        weight = math.log10(1+((word_occurrences_in_level/total_words_in_level)/(word_occurrences_in_corpus/total_words_in_corpus)))
    except ZeroDivisionError:
        weight = math.log10(1)
    return weight

Vengono costruiti i vettori:

In [ ]:
def build_vectors(bows,save_to_file=True,file_name="vectors"):
  labels = [label for label in bows.keys() if label != "full_corpus"] #Filtriamo le label con cui costruire i vettori (escludendo quindi "full_corpus")

  vectors = {}

  all_unique_words = set(bows["full_corpus"].keys()) #Raccogliamo tutte le parole uniche da full_corpus

  for i, word in enumerate(all_unique_words, start=1):
      if i % 1000 == 0:
          print(f"Vector n°{i} out of {len(all_unique_words)}")
      vectors[word] = [compute_weight(bows, level, word) for level in labels]
  print("Vectors built")

  if save_to_file:
    os.makedirs('/content/lexical_analysis/vectors',exist_ok=True)
    with open(os.path.join('/content/lexical_analysis/vectors', f"vectors_{file_name}.json"),mode="w",encoding="utf-8") as file:
        json.dump(vectors,file,indent=4)
  return vectors

Viene creato questo dizionario di combinazioni nel quale verranno inseriti i vettori.

In [ ]:
lexical_weight_extraction_vectors = {
    "0.raw":{},
    "1.lowercased":{},
    "2.lowercased_lemmatized":{},
    "3.lowercased_nostopwords":{},
    "4.lowercased_nostopwords_lemmatized":{},
    "5.lowercased_nopunctuation":{},
    "6.lowercased_nopunctuation_lemmatized":{},
    "7.lowercased_nopunctuation_nostopwords":{},
    "8.lowercased_nopunctuation_nostopwords_lemmatized":{}
}

Dato che ci vogliono circa 23 minuti per trasformare tutte le combinazioni in vettori, queste possono essere importate tramite la cella sottostante:

In [ ]:
for filename in os.listdir("/content/material_notebook_2/lexical_analysis/vectors"):
    if filename.startswith("vectors_") and filename.endswith(".json"):
        key = filename.replace("vectors_", "").replace(".json", "")
        filepath = os.path.join("/content/material_notebook_2/lexical_analysis/vectors", filename)
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
            lexical_weight_extraction_vectors[key] = data
        print(f"Loaded vectors for: {key}")

Loaded vectors for: 4.lowercased_nostopwords_lemmatized
Loaded vectors for: 3.lowercased_nostopwords
Loaded vectors for: 5.lowercased_nopunctuation
Loaded vectors for: 1.lowercased
Loaded vectors for: 6.lowercased_nopunctuation_lemmatized
Loaded vectors for: 7.lowercased_nopunctuation_nostopwords
Loaded vectors for: 2.lowercased_lemmatized
Loaded vectors for: 0.raw
Loaded vectors for: 8.lowercased_nopunctuation_nostopwords_lemmatized


Alternativamente, questo ciclo *for* vengono compilate tutte le combinazioni (quella sottostante è la cella utilizzata per compilare i file scaricati nella cella precedente):

In [ ]:
def vectors_compiler(n_text_to_compile=None):
  global texts
  global lexical_weight_extraction_vectors
  for key in lexical_weight_extraction_vectors.keys():
    if n_text_to_compile is None:
      bows = create_bows(texts["main_dataset"][key]["train"], save_to_file=True, file_name=key)
    elif type(n_text_to_compile) == int:
      bows = create_bows(texts["main_dataset"][key]["train"][:n_text_to_compile], save_to_file=True, file_name=key)
    vectors = build_vectors(bows, save_to_file=True, file_name=key)
    lexical_weight_extraction_vectors[key] = vectors

In [ ]:
vectors_compiler(5)

BoW n°0 out of 5
Vectors built
BoW n°0 out of 5
Vectors built
BoW n°0 out of 5
Vectors built
BoW n°0 out of 5
Vectors built
BoW n°0 out of 5
Vectors built
BoW n°0 out of 5
Vectors built
BoW n°0 out of 5
Vectors built
BoW n°0 out of 5
Vectors built
BoW n°0 out of 5
Vectors built


Una volta creati i vettori, possiamo creare una funzione

```
weight_extraction_predict(vectors, text)
```


che prende come argomenti:
* vettori (ora salvati sia su file che nel dizionario "vectorized_combinations");
* testo da predire.

In [ ]:
def weight_extraction_predict(vectors, text):
    global word_filter

    bow = {}
    counter = {
        "__label__A1":0,
        "__label__A2":0,
        "__label__B1":0,
        "__label__B2":0,
        "__label__C1":0
        }

    for word in [w for w in word_tokenize(text) if w in word_filter]:
        if word in bow:
            bow[word] += 1
        else:
            bow[word] = 1
    for key in bow.keys():
        if key in vectors:
            word_occurences_in_text_to_predict = bow[key]
            counter["__label__A1"] += vectors[key][0]*word_occurences_in_text_to_predict
            counter["__label__A2"] += vectors[key][1]*word_occurences_in_text_to_predict
            counter["__label__B1"] += vectors[key][2]*word_occurences_in_text_to_predict
            counter["__label__B2"] += vectors[key][3]*word_occurences_in_text_to_predict
            counter["__label__C1"] += vectors[key][4]*word_occurences_in_text_to_predict
    return max(counter, key=counter.get)

Testiamo la funzione su un testo casuale (avendo cura di utilizzare la stessa combinazione di preprocessing):

In [ ]:
label, text_to_predict = random.choice(texts["main_dataset"]["2.lowercased_lemmatized"]["test"]).split(" ", maxsplit=1)
print(f"Etichetta Gold: '{label}' ===> Testo: '{text_to_predict}'")

prediction = weight_extraction_predict(lexical_weight_extraction_vectors["2.lowercased_lemmatized"], text_to_predict)
print(f"Predizione:     '{prediction}'")

Etichetta Gold: '__label__C1' ===> Testo: 'think successful person have an inner strength to be resilient in this mad world he have bright target that lead he to the top of success know one man who do everything to get everything he want what the main trait or maybe even habit he have be passionate he achieve his goal only if this target be interesting for he if that get he mad about it however after get his purpose he lose his interest in what he be crash just yesterday and after moment find new passionate that lead he to the new height all his way to success he be surround with his family and friend maybe they give he strength and desire to be what he be maybe he do all of that for his family do not know what really know his surrounding love he and admire he'
Predizione:     '__label__C1'


###**Formule di leggibilità**
Le **formule di leggibilità** sono delle metriche che, a partire da diversi parametri, calcolano un risultato che indica il livello di difficoltà di un testo. Nel nostro caso utilizzeremo 6 formule:
1. Flesch-Kincaid ;
2. Gunning-Fog ;
3. Coleman-Liau ;
4. ARI.

Per ognuna verranno calcolate l"F1-Score (per ogni etichetta) e l"Accuracy.

####**Primi step**

#####**Estrazione dei dati**
Le varie formule necessitano dei dati numerici e conteggi per eseguire i calcoli. Tramite la seguente funzione è possibile estrarre questi dati:

In [ ]:
def extract_readability_features(text):

    #Tokenizzazione
    words = word_tokenize(text)
    sentences = sent_tokenize(text)

    #Primi conteggi
    n_words = len(words)
    n_sentences = len(sentences)

    #Inizializziamo le variabili per altri conteggi utili
    total_syllables = 0
    complex_words = 0
    ARI_chars = 0
    n_letter_in_100_words = 0

    #Tramite una sola iterazione, possiamo estrarre dei dati preliminari dai quali estrarremo gli altri conteggi
    for i, word in enumerate(words):
        syl_count = syllables.estimate(word)
        total_syllables += syl_count #Totale delle sillabe
        if syl_count > 2:
            complex_words += 1 #Totale delle parole complesse
        ARI_chars += sum(c.isalnum() for c in word) #Totale dei caratteri per la formula ARI (cioè tutti i caratteri alfanumerici)
        if i < 100:
            n_letter_in_100_words += len(word) #Numero di parole in 100 lettere

    #Qui si calcolano dei dati successivi
    avg_syllables_per_word = total_syllables / n_words #Media di sillabe per parola
    avg_words_per_sentence = n_words / n_sentences #Media di parole per frase
    avg_letters_per_100_words = (n_letter_in_100_words / 100) if n_words >= 100 else n_letter_in_100_words #Media di lettere su 100 parole
    avg_sentences_per_100_words = (n_sentences / n_words) * 100 #Media di frasi su 100 parole
    number_of_syllables = total_syllables
    number_of_words = len(words) #Numero di parole
    number_of_sentences = len(sentences) #Numero di frasi
    number_of_complex_words = complex_words #Numero di parole complesse
    number_of_characters = ARI_chars #Numero di caratteri ARI
    ARI_number_of_words = text.count(" ") + 1  #Numero di parole ARI "(1 parola = 1 spazio) + 1"

    return {
        "average_syllables_per_word": avg_syllables_per_word,
        "average_words_per_sentence": avg_words_per_sentence,
        "number_of_syllables": number_of_syllables,
        "number_of_words": number_of_words,
        "number_of_sentences": number_of_sentences,
        "number_of_complex_words": number_of_complex_words,
        "average_letters_per_100_words": avg_letters_per_100_words,
        "average_sentences_per_100_words": avg_sentences_per_100_words,
        "number_of_characters": number_of_characters,
        "ARI_number_of_words": ARI_number_of_words
    }


#####**Conversione in 60esimi:**
Dato che lavoreremo con 6, definiamo una funzione che converta il tutto in sessantesimi:

In [ ]:
def convert_to_60_scale(x, min_val, max_val):
    if max_val == min_val:
        raise ValueError("min_val e max_val non possono essere uguali")
    return 1 + (x - min_val) * (59 / (max_val - min_val))

#####**Restituzione CEFR**
Creiamo una funzione che prende un valore in 60esimi e lo converte ad un'etichetta *CEFR*.

In [ ]:
def return_as_cefr(scaled_score):
  if scaled_score <= 10:
    return "__label__A1"
  elif scaled_score <= 20:
    return "__label__A2"
  elif scaled_score <= 30:
    return "__label__B1"
  elif scaled_score <= 40:
    return "__label__B2"
  else:
    return "__label__C1"

####**Formule e funzione "template"**
La funzione template è questa:
```
def compute_readability_formula(readability_features)->tuple:
  """
  Compute the FORMULA readability score for a given text.
  Returns:
    -tuple: ( 1.score,   2.score converted to x/60,    3.CEFR label)
  """
  try:
    score = formula

  except ZeroDivisionError:
    return (0, 0, '__label__NONE')
  converted_score = convert_to_60_scale(score,min:int,max:int)
  return (score, converted_score, return_as_cefr(converted_score))
```
Verrà usata per ognuna delle 5 formule, e verrà "personalizzata" di volta in volta, modificando:
1. Nome della formula di leggibilità;
2. Formula effettiva che restituisce lo *score* di leggibilità;
3. Valore minimo e massimo di conversione.

Viene restituita una tupla contenente 3 valori:
* Punteggio della formula;
* Punteggio convertito in 60esimi;
* Punteggio convertito a CEFR;

#####**1. Flesch-Kincaid**
$$
0.39 \left(\frac{\text{words}}{\text{sentences}}\right)
+ 11.8 \left(\frac{\text{syllables}}{\text{words}}\right)
- 15.59
$$

In [ ]:
def compute_flesch_kincaid(readability_features)->tuple:
  """
  Compute the Flesch-Kincaid readability score for a given text.
  Returns:
    -tuple: ( 1.score,   2.score converted to x/60,    3.CEFR label)
  """
  try:
    score = 0.39*(readability_features["number_of_words"]/readability_features["number_of_sentences"]) + 11.8*(readability_features["number_of_syllables"]/readability_features["number_of_words"]) - 15.59
  except ZeroDivisionError:
    return (0, 0, '__label__NONE')
  converted_score = convert_to_60_scale(score,1,18)
  return (score, converted_score, return_as_cefr(converted_score))

#####**2. Gunning-Fog**
$$
0.4 \left[
\left(\frac{\text{words}}{\text{sentences}}\right)
+ 100 \left(\frac{\text{complex words}}{\text{words}}\right)
\right]
$$

In [ ]:
def compute_gunning_fog(readability_features)->tuple:
  """
  Compute the Gunning Fog readability score for a given text.
  Returns:
    -tuple: ( 1.score,   2.score converted to x/60,    3.CEFR label)
  """
  try:
    score = 0.4*((readability_features["number_of_words"]/readability_features["number_of_sentences"]) + 100*(readability_features["number_of_complex_words"]/readability_features["number_of_words"]))
  except ZeroDivisionError:
    return (0, 0, '__label__NONE')
  converted_score = convert_to_60_scale(score,6,17)

  return (score, converted_score, return_as_cefr(converted_score))

#####**3. Coleman-Liau**
$$
0.0588L - 0.296S - 15.8
$$

In [ ]:
def compute_coleman_liau(readability_features)->tuple:
  """
  Compute the Coleman-Liau readability score for a given text.
  Returns:
    -tuple: ( 1.score,   2.score converted to x/60,    3.CEFR label)
  """
  try:
    score = 0.0588*readability_features["average_letters_per_100_words"] - 0.296*readability_features["average_sentences_per_100_words"] - 15.8
  except ZeroDivisionError:
    return (0, 0, '__label__NONE')
  converted_score = convert_to_60_scale(score,1,16)
  return (score, converted_score, return_as_cefr(converted_score))

#####**4. ARI**
$$
4.71 \left(\frac{\text{characters}}{\text{words}}\right)
+ 0.5 \left(\frac{\text{words}}{\text{sentences}}\right)
- 21.43
$$

In [ ]:
def compute_ari(text)->tuple:
  """
  Compute the ARI readability score for a given text.
  Returns:
    -tuple: ( 1.score,   2.score converted to x/60,    3.CEFR label)
  """
  try:
    score = 4.71*(readability_features["number_of_characters"]/readability_features["ARI_number_of_words"]) + 0.5*(readability_features["ARI_number_of_words"]/readability_features["number_of_sentences"]) - 21.43
  except ZeroDivisionError:
    return (0, 0, '__label__NONE')
  converted_score = convert_to_60_scale(score,1,14)
  return (score, converted_score, return_as_cefr(converted_score))

####**Testiamo le formule**

In [ ]:
label, text_to_predict = random.choice(texts["main_dataset"]["0.raw"]["test"]).split(" ", maxsplit=1)

readability_features = extract_readability_features(text_to_predict)

print(f"Etichetta Gold: '{label}' ===> Testo: '{text_to_predict}'\n")

print(f"Flesch-Kincaid Prediction: {compute_flesch_kincaid(readability_features)[-1]}")
print(f"Gunning-Fog Prediction: {compute_gunning_fog(readability_features)[-1]}")
print(f"Coleman-Liau Prediction: {compute_coleman_liau(readability_features)[-1]}")
print(f"ARI Prediction: {compute_ari(readability_features)[-1]}")

Etichetta Gold: '__label__A1' ===> Testo: 'ugly mancus! overtime mancus city is hot, spring,winter,autumn and in summer is very hot.i always wear shot and t-shirt.'

Flesch-Kincaid Prediction: __label__B1
Gunning-Fog Prediction: __label__A1
Coleman-Liau Prediction: __label__A1
ARI Prediction: __label__B1


##**Machine Learning**
Il secondo dei due approcci consiste nell'allenare vari modelli di Machine Learning supervisionato, in due modi:
* Estrazione manuale delle features;
* Estrazione automatica delle features (*Fasttext* ).


###**Feature Extraction**
Utilizzando *spacy*, vengono estratte manualmente varie features.

In [ ]:
vowel_pattern = re.compile(r'[aeiouyAEIOUY]{2,}') #creiamo un pattern di vocali con una RegEx, che trovi tutte le vocali doppie o più (ad esempio, "fAUlt")
modal_verbs = {"must","will","would","should","might","may","can","could"} #creiamo un set con vari verbi modali

In [ ]:
def feature_extraction(text):
    global connectors

    doc = nlp(text)

    #Tokenizzazione e conteggi iniziali
    tokens = [t.text for t in doc]
    n_tokens = len(tokens)
    token_counts = Counter(tokens)
    sentences = list(doc.sents)
    n_sent = len(sentences)

    #Calcolo delle metriche lessicali di base
    avg_word_freq = (sum(c / n_tokens for c in token_counts.values()) / len(token_counts)) if n_tokens else 0
    vocab_richness = len(token_counts) / n_tokens if n_tokens else 0
    avg_word_len = sum(len(t) for t in tokens) / n_tokens if n_tokens else 0

    # Conteggio delle parole polisillabiche
    multisyllable_count = sum(1 for t in tokens if len(vowel_pattern.findall(t)) >= 2)
    multisyllable_pct = (100 * multisyllable_count / n_tokens) if n_tokens else 0

    # Calcolo delle metriche a livello di frase
    avg_sent_len = (sum(len(s) for s in sentences) / n_sent) if n_sent else 0
    n_sentences_feat = n_sent # Numero totale di frasi

    # Conteggio delle clausole subordinate
    sub_clauses = sum(1 for t in doc if t.dep_ in {"advcl","ccomp","xcomp","acl","relcl"}) # Conta le clausole subordinate
    subordinate_pct = (100 * sub_clauses / n_sent) if n_sent else 0

    # Identificazione e conteggio di verbi modali, passati e connettivi
    token_set = set(t.lower() for t in tokens)
    modal_pct = (100 * len(token_set & modal_verbs) / n_tokens) if n_tokens else 0 # Percentuale di verbi modali

    past_verbs = sum(1 for t in doc if t.tag_ in {"VBD","VBN"}) # Conta i verbi al passato e i participi passati
    past_pct = (100 * past_verbs / n_tokens) if n_tokens else 0

    connectors_pct = (100 * len(token_set & connectors) / n_tokens) if n_tokens else 0 # Percentuale di connettivi discorsivi

    # Conteggio della punteggiatura
    punct_count = sum(1 for t in tokens if t in ".:,;?!-") # Conta i segni di punteggiatura
    punct_pct = (100 * punct_count / n_tokens) if n_tokens else 0

    return [
        avg_word_freq,
        vocab_richness,
        avg_word_len,
        multisyllable_pct,
        avg_sent_len,
        n_sentences_feat,
        subordinate_pct,
        modal_pct,
        past_pct,
        connectors_pct,
        punct_pct
    ]

Vediamo 3 testi a caso vettorizzati

In [ ]:
samples = random.choices(texts["main_dataset"]["0.raw"]["test"], k=3)
for sample in samples:
    print(f"Etichetta ===> {sample.split(' ',1)[0]}")
    print(f"Testo     ===> {sample.split(' ',1)[1]}")
    print(f"Features  ===> {feature_extraction(sample)}\n")

Etichetta ===> __label__C1
Testo     ===> We'll be going to have our task done in an effective way by way of changing the sequences of procedure. In that case, we can do things without any redundancy, even postponement. By examining the procedures, we also can easily find out what goes wrong or what has gone wrong and have it corrected in no time. That's how efficiency produce in an company. The ways you can start with are to make a list what you want to do and will do in order of importance. arrange them according to your cognitive judgement on your proficiency. That won't be easy at the first beginning, however, that will give you a change to think and reflect the good outcome on your work.
Features  ===> [0.010638297872340425, 0.6714285714285714, 3.9214285714285713, 0.0, 20.0, 7, 142.85714285714286, 1.4285714285714286, 2.142857142857143, 5.714285714285714, 8.571428571428571]

Etichetta ===> __label__B2
Testo     ===> Hi, Ian, I heard you would leave us due to your phobia of enclosed

####**Vettorizzazione dei testi**
Prima di tutto va creata la directory per conservare i testi:

In [ ]:
base_dir = Path("/content/vectorized_texts")
(base_dir / "main_dataset").mkdir(parents=True, exist_ok=True)
(base_dir / "generalized_dataset").mkdir(parents=True, exist_ok=True)

Le features verranno salvate in 9 (+ 9) file diversi, dove ogni file sarà un JSON con la seguente struttura:


  ```
  {
    "train":{
      "labels":[2, 3, ...],
      "texts_as_vectors":[
        [104, 11, 0.0136986301369863, 0.7019230769230769, 3.6346153846153846, 11, 10.576923076923077, 9.454545454545455, 8, 72.72727272727273, 2.8846153846153846, 11, 10.576923076923077, 11, 10.576923076923077],
        [37, 4, 0.038461538461538464, 0.7027027027027027, 3.5405405405405403, 4, 10.81081081081081, 9.25, 0, 0.0, 0.0, 0, 0.0, 7, 18.91891891891892],
        ...,
      ]
    },
    "test":{
      "labels":[1, 4, ...],
      "texts_as_vectors":[
        [139, 11, 0.01020408163265306, 0.7050359712230215, 3.949640287769784, 17, 12.23021582733813, 12.636363636363637, 7, 63.63636363636363, 1.4388489208633093, 11, 7.913669064748201, 16, 11.510791366906474],
        [199, 10, 0.008130081300813009, 0.6180904522613065, 4.251256281407035, 25, 12.56281407035176, 19.9, 14, 140.0, 2.0100502512562812, 1, 0.5025125628140703, 15, 7.5376884422110555],
        ...,
      ]
    }
  }
  ```



Dato che i modelli di SkLearn non possono lavorare con etichette formate da stringhe, creiamo due dizionari che convertano le etichette a numeri, e viceversa:
* *cefr_to_int* ;
* *int_to_cefr* ;

In [ ]:
cefr_to_int = {
        "__label__A1": 0,
        "__label__A2": 1,
        "__label__B1": 2,
        "__label__B2": 3,
        "__label__C1": 4
}

int_to_cefr = {
        0: "__label__A1",
        1: "__label__A2",
        2: "__label__B1",
        3: "__label__B2",
        4: "__label__C1"
}

E qui creiamo la funzione per costruire i vettori di tutti i testi

In [ ]:
def build_text_vectors(n_texts_to_vectorize=None, save_to_file=True, do_print=False):
  for dataset_type in texts.keys():
      for preprocessing_type in texts[dataset_type].keys():
        structure = {
                'train':{'labels':[],'texts_as_vectors':[]},
                'test':{'labels':[],'texts_as_vectors':[]}
                         }
        for split in [sp for sp in texts[dataset_type][preprocessing_type].keys() if sp in ('train','test')]:

          if n_texts_to_vectorize is not None:
            texts_to_work_on = texts[dataset_type][preprocessing_type][split][:n_texts_to_vectorize]
          else:
            texts_to_work_on = texts[dataset_type][preprocessing_type][split]

          for text in texts_to_work_on:
            label, text = text.split(' ',1)

            label = cefr_to_int[label]
            features = feature_extraction(text)

            structure[split]['labels'].append(label)
            structure[split]['texts_as_vectors'].append(features)
        if do_print:
          print(dataset_type,preprocessing_type)
          print(structure)
        if save_to_file:
          os.makedirs('vectorized_texts',exist_ok=True)
          with open(os.path.join('/content/vectorized_texts',dataset_type, f'vectorized_{preprocessing_type}.json'), 'w', encoding='utf-8') as f:
            json.dump(structure, f, ensure_ascii=False, indent=4)

Testiamo la funzione su 5 testi per sezione:

In [ ]:
build_text_vectors(5,save_to_file=True, do_print=True)

main_dataset 0.raw
{'train': {'labels': [0, 1, 0, 0, 4], 'texts_as_vectors': [[0.029411764705882353, 0.7391304347826086, 3.260869565217391, 0.0, 15.333333333333334, 3, 133.33333333333334, 0.0, 0.0, 8.695652173913043, 17.391304347826086], [0.02702702702702703, 0.8222222222222222, 4.155555555555556, 0.0, 11.25, 4, 25.0, 4.444444444444445, 8.88888888888889, 6.666666666666667, 8.88888888888889], [0.022727272727272728, 0.7857142857142857, 4.089285714285714, 3.5714285714285716, 8.0, 7, 14.285714285714286, 0.0, 0.0, 8.928571428571429, 16.071428571428573], [0.029411764705882353, 0.723404255319149, 4.0, 0.0, 47.0, 1, 100.0, 0.0, 0.0, 4.25531914893617, 29.78723404255319], [0.010416666666666666, 0.588957055214724, 4.282208588957055, 0.0, 13.583333333333334, 12, 50.0, 0.6134969325153374, 3.6809815950920246, 3.067484662576687, 7.361963190184049]]}, 'test': {'labels': [4, 1, 3, 0, 0], 'texts_as_vectors': [[0.008695652173913044, 0.5637254901960784, 3.6862745098039214, 0.0, 15.692307692307692, 13, 92.

Dato che l'estrazione di features su quasi 500.000 testi richiede varie ore, è possibile caricare i vettori nella variabile *vectorized_texts* a partire da vari file *JSON* tramite la cella sottostante:

In [ ]:
vectorized_texts = {}
base_dir_path = '/content/material_notebook_2/other_stuff/vectorized_texts'

for dataset_type_folder in [folder for folder in os.listdir(base_dir_path) if not folder.endswith('ipynb_checkpoints')]:
  print(f'Dataset: "{dataset_type_folder}"')
  dataset_type_path = os.path.join(base_dir_path, dataset_type_folder)

  vectorized_texts[dataset_type_folder] = {}

  for filename in [f for f in os.listdir(dataset_type_path) if f.startswith('vectorized_') and f.endswith('.json')]:
    preprocessing_type = filename.replace('vectorized_', '').replace('.json', '')
    filepath = os.path.join(dataset_type_path, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
        vectorized_texts[dataset_type_folder][preprocessing_type] = data
    print(f'    File: "{filename}" loaded')

Dataset: "main_dataset"
    File: "vectorized_4.lowercased_nostopwords_lemmatized.json" loaded
    File: "vectorized_7.lowercased_nopunctuation_nostopwords.json" loaded
    File: "vectorized_1.lowercased.json" loaded
    File: "vectorized_6.lowercased_nopunctuation_lemmatized.json" loaded
    File: "vectorized_8.lowercased_nopunctuation_nostopwords_lemmatized.json" loaded
    File: "vectorized_0.raw.json" loaded
    File: "vectorized_5.lowercased_nopunctuation.json" loaded
    File: "vectorized_3.lowercased_nostopwords.json" loaded
    File: "vectorized_2.lowercased_lemmatized.json" loaded
Dataset: "generalized_dataset"
    File: "vectorized_4.lowercased_nostopwords_lemmatized.json" loaded
    File: "vectorized_7.lowercased_nopunctuation_nostopwords.json" loaded
    File: "vectorized_1.lowercased.json" loaded
    File: "vectorized_6.lowercased_nopunctuation_lemmatized.json" loaded
    File: "vectorized_8.lowercased_nopunctuation_nostopwords_lemmatized.json" loaded
    File: "vectorized

###**Modelli classici**

Creiamo inanzitutto una funzione generale che prenda come parametri il modello da usare e il testo, che può essere:
1. Testo effettivo, da rendere un vettore;
2. Testo vettorizzato, semplicemente da predire.

Possiamo scegliere tra il testo effettivo e il testo vettorizzato per motivi di tempo: infatti, vettorizzare e predire molti testi richiede molto più tempo rispetto alla semplice predizione, quindi, in fase di testing, verranno usati per le predizioni i vettori scaricati precedentemente.

In [ ]:
def traditional_model_predict(model, text):
  if type(text) == str:
    features = feature_extraction(text)
  else:
    features = text
  return int_to_cefr[model.predict([features])[0]]

Estraiamo da *vectorized_texts* i due dataset "0.raw" (*texts_as_vectors* e *labels*) e assegniamoli alle due variabili *training_vectors* e *training labels*:

In [ ]:
training_vectors, training_labels = vectorized_texts["main_dataset"]["0.raw"]["train"]["texts_as_vectors"], vectorized_texts["main_dataset"]["0.raw"]["train"]["labels"]

####**Random Forest**

Definiamo una prima funzione per allenare il modello:

In [ ]:
def train_random_forest(train, train_labels):
    clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
    )

    clf.fit(train, train_labels)
    return clf

Alleniamo, per ora, il modello sulla split *0.raw* :

In [ ]:
random_forest_model = train_random_forest(training_vectors,training_labels)

Creiamo ora una funzione che prende come parametro un testo, lo vettorizza e produce una previsione:

In [ ]:
for i in range(10):
  gold_label, text_to_predict = random.choice(texts["main_dataset"]["0.raw"]["test"]).split(" ",1)
  print(f'Testo {i+1}')
  print(f'  Etichetta gold     ===> {gold_label}')
  print(f'  Etichetta predetta ===> {traditional_model_predict(random_forest_model,text_to_predict)}\n')

Testo 1
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__B2

Testo 2
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 3
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__B2

Testo 4
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 5
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__B2

Testo 6
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__A2

Testo 7
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 8
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1

Testo 9
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 10
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A2



Testiamo invece la funzione 10 testi già vettorizzati

In [ ]:
for i in range(10):
  print(f'Testo {i+1}')
  gold_label, text_to_predict = vectorized_texts["main_dataset"]["0.raw"]["test"]['labels'][i], vectorized_texts["main_dataset"]["0.raw"]["test"]['texts_as_vectors'][i]
  print(f'  Etichetta gold     ===> {int_to_cefr[gold_label]}')
  print(f'  Etichetta predetta ===> {traditional_model_predict(random_forest_model,text_to_predict)}\n')

Testo 1
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__C1

Testo 2
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1

Testo 3
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__B2

Testo 4
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 5
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 6
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__C1

Testo 7
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 8
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__A2

Testo 9
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__B1

Testo 10
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A2



Alleniamo tutti i modelli Random Forest (servono circa 3/4 minuti). Verranno usati per le predizioni alla fine del notebook.

In [ ]:
random_forest_models = {}

for preprocessing_type in vectorized_texts["main_dataset"].keys():
  training_vectors = vectorized_texts["main_dataset"][preprocessing_type]["train"]["texts_as_vectors"]
  training_labels = vectorized_texts["main_dataset"][preprocessing_type]["train"]["labels"]
  model = train_random_forest(training_vectors, training_labels)
  random_forest_models[preprocessing_type] = model
  print(f"Finished training for {preprocessing_type}")

Finished training for 4.lowercased_nostopwords_lemmatized
Finished training for 7.lowercased_nopunctuation_nostopwords
Finished training for 1.lowercased


KeyboardInterrupt: 

####**Logistic Regression**
Definiamo una funzione per la Logistic Regression

In [ ]:
def train_logistic_regression(train, train_labels):
  clf = Pipeline([
      ("scaler", StandardScaler()),
      ("logreg", LogisticRegression(
          max_iter=2000,
          solver='lbfgs',
          penalty='l2',
          C=1.0,
          random_state=42
      ))
  ])

  clf.fit(train, train_labels)
  return clf

Alleniamo un modello, come per il Random Forest

In [ ]:
log_regression_model = train_logistic_regression(training_vectors,training_labels)

E lo testiamo su i testi da vettorizzare

In [ ]:
for i in range(10):
  print(f'Testo {i+1}')
  gold_label, text_to_predict = random.choice(texts["main_dataset"]["0.raw"]["test"]).split(" ",1)
  print(f'  Etichetta gold     ===> {gold_label}')
  print(f'  Etichetta predetta ===> {traditional_model_predict(log_regression_model,text_to_predict)}\n')

Testo 1
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A2

Testo 2
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 3
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__A2

Testo 4
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 5
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 6
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__A2

Testo 7
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Testo 8
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__A2

Testo 9
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1

Testo 10
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__A2



E anche su 10 testi vettorizzati

In [ ]:
for i in range(10):
  gold_label, text_to_predict = vectorized_texts["main_dataset"]["0.raw"]["test"]['labels'][i], vectorized_texts["main_dataset"]["0.raw"]["test"]['texts_as_vectors'][i]
  print(f'Text {i+1}')
  print(f'  Etichetta gold     ===> {int_to_cefr[gold_label]}')
  print(f'  Etichetta predetta ===> {traditional_model_predict(log_regression_model,text_to_predict)}\n')

Text 1
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__A2

Text 2
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1

Text 3
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__A2

Text 4
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A2

Text 5
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Text 6
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__A2

Text 7
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1

Text 8
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__A1

Text 9
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__A2

Text 10
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1



Alleniamo tutti e 9 i modelli:

In [ ]:
logistic_regression_models = {}

for preprocessing_type in vectorized_texts["main_dataset"].keys():
  training_vectors = vectorized_texts["main_dataset"][preprocessing_type]["train"]["texts_as_vectors"]
  training_labels = vectorized_texts["main_dataset"][preprocessing_type]["train"]["labels"]
  model = train_logistic_regression(training_vectors, training_labels)
  logistic_regression_models[preprocessing_type] = model
  print(f"Finished training for {preprocessing_type}")

Finished training for 4.lowercased_nostopwords_lemmatized
Finished training for 7.lowercased_nopunctuation_nostopwords
Finished training for 1.lowercased
Finished training for 6.lowercased_nopunctuation_lemmatized
Finished training for 8.lowercased_nopunctuation_nostopwords_lemmatized
Finished training for 0.raw
Finished training for 5.lowercased_nopunctuation
Finished training for 3.lowercased_nostopwords
Finished training for 2.lowercased_lemmatized


####**Gradient Boosting**
Anche qui definiamo un modello:

In [ ]:
def train_gradient_boosting(train, train_labels):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("gb", GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ))
    ])
    clf.fit(train, train_labels)
    return clf

Lo alleniamo:

In [ ]:
grad_boosting_model = train_gradient_boosting(training_vectors,training_labels)

E lo testiamo su 10 testi a caso da vettorizzare:

In [ ]:
for i in range(10):
  gold_label, text_to_predict = random.choice(texts["main_dataset"]["0.raw"]["test"]).split(" ",1)
  print(f'Testo {i+1}')
  print(f'  Etichetta gold     ===> {gold_label}')
  print(f'  Etichetta predetta ===> {traditional_model_predict(grad_boosting_model,text_to_predict)}\n')

Testo 1
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__B1

Testo 2
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__B1

Testo 3
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__B1

Testo 4
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__B1

Testo 5
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__B1

Testo 6
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1

Testo 7
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1

Testo 8
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__A1

Testo 9
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__B1

Testo 10
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1



E su 10 testi già vettorizzati:

In [ ]:
for i in range(10):
  gold_label, text_to_predict = vectorized_texts["main_dataset"]["0.raw"]["test"]['labels'][i], vectorized_texts["main_dataset"]["0.raw"]["test"]['texts_as_vectors'][i]
  print(f'Testo {i+1}')
  print(f'  Etichetta gold     ===> {int_to_cefr[gold_label]}')
  print(f'  Etichetta predetta ===> {traditional_model_predict(grad_boosting_model,text_to_predict)}')

Testo 1
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__A1
Testo 2
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1
Testo 3
  Etichetta gold     ===> __label__B2
  Etichetta predetta ===> __label__C1
Testo 4
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A2
Testo 5
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1
Testo 6
  Etichetta gold     ===> __label__C1
  Etichetta predetta ===> __label__B1
Testo 7
  Etichetta gold     ===> __label__A1
  Etichetta predetta ===> __label__A1
Testo 8
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__A2
Testo 9
  Etichetta gold     ===> __label__B1
  Etichetta predetta ===> __label__A1
Testo 10
  Etichetta gold     ===> __label__A2
  Etichetta predetta ===> __label__A1


Alleniamo tutti i modelli (servono circa 10 minuti):

In [ ]:
gradient_boosting_models = {}

for preprocessing_type in vectorized_texts["main_dataset"].keys():
    training_vectors = vectorized_texts["main_dataset"][preprocessing_type]["train"]["texts_as_vectors"]
    training_labels = vectorized_texts["main_dataset"][preprocessing_type]["train"]["labels"]
    model = train_gradient_boosting(training_vectors, training_labels)
    gradient_boosting_models[preprocessing_type] = model
    print(f"Finished training for {preprocessing_type}")

###**Fasttext**
Tramite Fasttext è possibile allenare un modello (in questo caso 9 modelli, 1 per ogni combinazione) e salvarlo in una cartella:

In [ ]:
def train_fasttext_models():
  parent_path = r"/content/material_notebook_2/preprocessed/main_dataset"

  os.makedirs("/content/fasttext_models", exist_ok=True)
  for subfolder in os.listdir(parent_path):
    training_path = os.path.join(parent_path, subfolder,"main_dataset.train")

    model_name = f"{subfolder}_fasttext_model.bin"
    print(f"Training model: {model_name}")

    model = fasttext.train_supervised(
        input=training_path,
        lr=0.5,
        epoch=25,
        wordNgrams=2,
        bucket=200000,
        dim=50,
        loss="ova"
        )

    model.save_model(os.path.join("/content/fasttext_models",model_name))
    print(f"{model_name} ==> Saved!","\n")

Dato che allenare tutti i modelli richiede circa 6-7 minuti...

In [ ]:
#train_fasttext_models()

Possiamo importarne uno, per ora (poi verranno tutti importati durante la fase di testing alla fine del notebook)

In [ ]:
model = fasttext.load_model(r"/content/material_notebook_2/other_stuff/fasttext_models/8.lowercased_nopunctuation_nostopwords_lemmatized_fasttext_model.bin")

Lo testiamo:

In [ ]:
for i in range(10):
  label, text = random.choice(texts["main_dataset"]["8.lowercased_nopunctuation_nostopwords_lemmatized"]["test"]).split(" ",maxsplit=1)
  print(f'Testo {i+1}')
  print(f"  Etichetta Gold: {label} ===> Testo: '{text}'")
  print(f"  Predizione:     {model.predict([text])[0][0][0]}\n")

Testo 1
  Etichetta Gold: __label__B2 ===> Testo: 'hi anna analyse zookeepers job opinion amazing believe urge accept exactly look deal different kind animal outside far salary absolutely good consider not experience area totally convinced great opportunity ll happy option completely well imagine work secretary way wait job happy fabiano'
  Predizione:     __label__B2

Testo 2
  Etichetta Gold: __label__A1 ===> Testo: 'hi name alex kiang china love listen music nice meet good eye alex'
  Predizione:     __label__A1

Testo 3
  Etichetta Gold: __label__B1 ===> Testo: 'dream tobe air pilot airline commercial train train ingenie melanic ipn mexico city guess lot similarity career help course career air pilot fast hope new year train air pilot meet job want work air pilot travel world year work unity rescue teacher tell seek correction write thank'
  Predizione:     __label__B1

Testo 4
  Etichetta Gold: __label__C1 ===> Testo: 'dear mayor bleak write express concern new supermart store pla

##**LLM**
Utilizzando l'API di Gemini 2.5 Flash, è possibile testare la capacità del modello nel predire il livello linguistico. Dato che le richieste giornaliere effettuabili sono molto limitate, le predizioni sono state testate solamente su circa 3000 testi, appartenenti alla combinazione preprocessing "raw" (ovvero testi dove non è stato rimosso nulla, se non i ritorni a capo), più precisamente:

1. **1493 predizioni *Zero-Shot*** : ovvero con un prompt privo di esempi;
2. **1493 predizioni *Few-Shot*** : ovvero con un prompt con 5 esempi per livello, ricavati dal dataset di training (che Gemini non ha mai visto);

In [ ]:
from google.colab import userdata
api_key = userdata.get("GOOGLE_API_KEY")

Definiamo inanzitutto una funzione per predire:

In [ ]:
def gemini_predictor(prompt_template, text):
    client = genai.Client(api_key=api_key)

    prompt = prompt_template.format(text=text)

    try:
      response = client.models.generate_content(
          model="gemini-2.5-flash",
          contents=prompt
      )
    except Exception as e:
      return f"Invalid_Response, Exception {e}"

    prediction = response.text.strip().upper()

    if prediction not in ["A1", "A2", "B1", "B2", "C1"]:
        return "Invalid_Response"
    else:
        return f"__label__{prediction}"


###**Zero-shot**
Definiamo il prompt *zero-shot*:

In [ ]:
text = ""
prompt_zero_shot = f"""You are an English language evaluator.
            Classify the CEFR level (A1, A2, B1, B2, or C1) of the following text.

            Text:
            {text}

            Reply with only one of these labels: A1, A2, B1, B2, or C1.
            Do not include explanations or any additional text."""

Testiamolo su un testo casuale:

In [ ]:
label, text = random.choice(texts["main_dataset"]["0.raw"]["test"]).split(" ",maxsplit=1)
print(f"Etichetta Gold: {label} ===> Testo: '{text}")
print(f"Predizione:     {gemini_predictor(prompt_zero_shot, text)}")

Etichetta Gold: __label__B1 ===> Testo: 'Hi,Tim I'm glad to hear that you have had a trilling but great vacation. Me too, I just went back last week . I and my friend Jane went to fijian Yunnan Province. I think it's a beautiful place. We took a boat in the river. The water was very clear ,we could see fish was swimming under the water. One morning, the weather was nice,the air smelled fresh, we walked in the forest. Birds were singing on the trees, the sun shined on the trees. we all enjoyed it. suddenly, I felt that I had stepped something softly. "snake" I jumped up and shouted. Fortunately, the snake didn't move to us, we ran away as quickly as possible. It's terrible. But the place is great. See you, Lucy
Predizione:     __label__A1


###**Few-shot**
Creiamo ora la sezione *Few-Shot*.
Prima di tutto viene creato un dizionario che conterrà gli esempi per il metodo *few-shot*:

In [ ]:
few_shots = {
    "__label__A1":[],
    "__label__A2":[],
    "__label__B1":[],
    "__label__B2":[],
    "__label__C1":[]
}

for pair in texts["main_dataset"]["0.raw"]["train"]:
  label, text = pair.split(" ",maxsplit=1)
  if len(few_shots[label]) < 5:
    few_shots[label].append(text)

Viene creato un prompt:

In [ ]:
text = ""
prompt_few_shot = f"""
            You are an English language evaluator.
            Classify the CEFR level (A1, A2, B1, B2, or C1) of the following text.

            Here are some examples for each level:

            A1 examples:
            - {few_shots["__label__A1"][0]}
            - {few_shots["__label__A1"][1]}
            - {few_shots["__label__A1"][2]}
            - {few_shots["__label__A1"][3]}
            - {few_shots["__label__A1"][4]}

            A2 examples:
            - {few_shots["__label__A2"][0]}
            - {few_shots["__label__A2"][1]}
            - {few_shots["__label__A2"][2]}
            - {few_shots["__label__A2"][3]}
            - {few_shots["__label__A2"][4]}

            B1 examples:
            - {few_shots["__label__B1"][0]}
            - {few_shots["__label__B1"][1]}
            - {few_shots["__label__B1"][2]}
            - {few_shots["__label__B1"][3]}
            - {few_shots["__label__B1"][4]}

            B2 examples:
            - {few_shots["__label__B2"][0]}
            - {few_shots["__label__B2"][1]}
            - {few_shots["__label__B2"][2]}
            - {few_shots["__label__B2"][3]}
            - {few_shots["__label__B2"][4]}

            C1 examples:
            - {few_shots["__label__C1"][0]}
            - {few_shots["__label__C1"][1]}
            - {few_shots["__label__C1"][2]}
            - {few_shots["__label__C1"][3]}
            - {few_shots["__label__C1"][4]}

            Text to evaluate:
            {text}

            Reply with only one of these labels: A1, A2, B1, B2, or C1.
            Do not include explanations or any additional text.
            """

In [ ]:
label, text = random.choice(texts["main_dataset"]["0.raw"]["test"]).split(" ",maxsplit=1)
print(f"Etichetta Gold:          {label} ===> Testo: '{text}'")
print(f"Predizione Few-Shot:     {gemini_predictor(prompt_few_shot, text)}")

Etichetta Gold:          __label__C1 ===> Testo: 'At 6:45 am, the police received a phone call from a woman who was screaming hysterically. They could not understand what she was saying so they asked her to calm down. The woman, who seemed to be in her late 50s, stopped screaming and spoke in a calm voice. She told the policeman that her little three-month-old black and white cat had jumped from her balcony on the third floor and was now stuck in a tree outside her window. The patient policeman was telling the now somewhat calmer lady not to worry; he would send a team to rescue the cat. The woman was reassured. finally, she told the policeman that it was important for them to know that it was the tall fir tree outside her bedroom window the cat was sitting in, not the pine tree outside her balcony.'
Predizione Few-Shot:     __label__A2


##**Testing preliminare**

###Struttura dati per i risultati
Definiamo due strutture dati per contenere tutti i classification report

In [ ]:
main_results = {
    "rule_based":{
        "lexical_analysis":{
            "raw_counts":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}
            },
            "weight_extraction":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}
            }
            },
        "readability":{
            "flesch_kincaid":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "gunning_fog":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "coleman_liau":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "ARI":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}}
        }
                  },
    "machine_learning":{
        "traditional":{
            "random_forest":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "logistic_regression":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "gradient_boosting":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}}
        },
        "fasttext":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}
        }
    },
    "LLM":{
        "gemini":{
            "zero_shot":{
                "0.raw":{},
        },
            "few_shot":{
                "0.raw":{},
        }
    }
}
}

In [ ]:
generalized_results = {
    "rule_based":{
        "lexical_analysis":{
            "raw_counts":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}
            },
            "weight_extraction":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}
            }
            },
        "readability":{
            "flesch_kincaid":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "gunning_fog":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "coleman_liau":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "ARI":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}}
        }
                  },
    "machine_learning":{
        "traditional":{
            "random_forest":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "logistic_regression":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}},
            "gradient_boosting":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}}
        },
        "fasttext":{
                "0.raw":{},
                "1.lowercased":{},
                "2.lowercased_lemmatized":{},
                "3.lowercased_nostopwords":{},
                "4.lowercased_nostopwords_lemmatized":{},
                "5.lowercased_nopunctuation":{},
                "6.lowercased_nopunctuation_lemmatized":{},
                "7.lowercased_nopunctuation_nostopwords":{},
                "8.lowercased_nopunctuation_nostopwords_lemmatized":{}
        }
        }
    }

Per ogni metodo andremo a riempire le due strutture dati con i classification report su ogni tipo di preprocessing.
Dato che il numero di predizioni fatte sui testi con Gemini sono state 1493, possiamo creare una variabile "indice" che permetta di calcolare le predizioni su esattamente 1493 testi:

In [ ]:
index_of_texts_to_predict = 1493

###**Rule-based**

####**Lexical Analysis**

#####**Raw Counts**

In [ ]:
for key in texts['main_dataset']:

  main_results['rule_based']['lexical_analysis']['raw_counts'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Preprocessing type: {key}')
  for text in texts['main_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1

    label, text = text.split(" ",maxsplit=1)
    prediction = raw_counts_prediction(text)

    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')

    gold.append(label)
    predictions.append(prediction)
  main_results['rule_based']['lexical_analysis']['raw_counts'][key] = classification_report(gold, predictions, output_dict=True,zero_division=1)

Preprocessing type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 1.lowercased
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 2.lowercased_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 3.lowercased_nostopwords
Predicting text 500 out of 1493
Predicting text 1000 out of 1493


KeyboardInterrupt: 

In [ ]:
for key in texts['generalized_dataset']:

  generalized_results['rule_based']['lexical_analysis']['raw_counts'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Preprocessing type: {key}')
  for text in texts['generalized_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1

    try:
      label, text = text.split(" ",maxsplit=1)
      prediction = raw_counts_prediction(text)
    except ValueError:
      counter = counter - 1
      continue

    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')

    gold.append(label)
    predictions.append(prediction)
  generalized_results['rule_based']['lexical_analysis']['raw_counts'][key] = classification_report(gold, predictions, output_dict=True,zero_division=1)

Preprocessing type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 1.lowercased
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 2.lowercased_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 3.lowercased_nostopwords
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 4.lowercased_nostopwords_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 5.lowercased_nopunctuation
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 6.lowercased_nopunctuation_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 7.lowercased_nopunctuation_nostopwords
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Preprocessing type: 8.lowercased_nopunctuation_nostopwords_lemmatized
Predicting text 500 out of 1493
Predicting

#####**Weight Extraction**

In [ ]:
for key in texts['main_dataset']:

  main_results['rule_based']['lexical_analysis']['weight_extraction'][key] = {}

  with open(f'/content/material_notebook_2/lexical_analysis/vectors/vectors_{key}.json','r', encoding='utf-8') as vec_file:
    vecs = json.load(vec_file)

  gold = []
  predictions = []
  counter = 0
  print(f'Type: {key}')
  for text in texts['main_dataset'][key]['test'][:index_of_texts_to_predict]:

    label, text = text.split(" ",maxsplit=1)
    prediction = weight_extraction_predict(vecs, text)

    counter = counter + 1
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')

    gold.append(label)
    predictions.append(prediction)
  main_results['rule_based']['lexical_analysis']['weight_extraction'][key] = classification_report(gold, predictions, output_dict=True)

Type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 1.lowercased
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 2.lowercased_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 3.lowercased_nostopwords
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 4.lowercased_nostopwords_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 5.lowercased_nopunctuation
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 6.lowercased_nopunctuation_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 7.lowercased_nopunctuation_nostopwords
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 8.lowercased_nopunctuation_nostopwords_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493


In [ ]:
for key in texts['generalized_dataset']:

  generalized_results['rule_based']['lexical_analysis']['weight_extraction'][key] = {}

  with open(f'/content/material_notebook_2/lexical_analysis/vectors/vectors_{key}.json','r', encoding='utf-8') as vec_file:
    vecs = json.load(vec_file)

  gold = []
  predictions = []
  counter = 0
  print(f'Type: {key}')
  for text in texts['generalized_dataset'][key]['test'][:index_of_texts_to_predict]:

    try:
      label, text = text.split(" ",maxsplit=1)
      prediction = weight_extraction_predict(vecs, text)
    except ValueError:
      counter = counter - 1
      continue

    counter = counter + 1
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')

    gold.append(label)
    predictions.append(prediction)
  generalized_results['rule_based']['lexical_analysis']['weight_extraction'][key] = classification_report(gold, predictions, output_dict=True)

Type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 1.lowercased
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 2.lowercased_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 3.lowercased_nostopwords
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 4.lowercased_nostopwords_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 5.lowercased_nopunctuation
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 6.lowercased_nopunctuation_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 7.lowercased_nopunctuation_nostopwords
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 8.lowercased_nopunctuation_nostopwords_lemmatized
Predicting text 500 out of 1493
Predicting text 1000 out of 1493


####**Formule di leggibilità**

#####**Flesch-Kincaid**

######**Main results**

In [ ]:
for key in texts['main_dataset']:
  main_results['rule_based']['readability']['flesch_kincaid'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Type: {key}')
  for text in texts['main_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1
    try:
      label, text = text.split(" ",maxsplit=1)
    except ValueError:
      counter = counter - 1
      continue

    readability_features = extract_readability_features(text)
    prediction = compute_flesch_kincaid(readability_features)[-1]

    gold.append(label)
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')
  main_results['rule_based']['readability']['flesch_kincaid'][key] = classification_report(gold, predictions, output_dict=True, zero_division=1)

Type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 1.lowercased


KeyboardInterrupt: 

######**Generalized results**

In [ ]:
for key in texts['generalized_dataset']:
  generalized_results['rule_based']['readability']['flesch_kincaid'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Type: {key}')
  for text in texts['generalized_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1
    try:
      label, text = text.split(" ",maxsplit=1)
    except ValueError:
      counter = counter - 1
      continue

    readability_features = extract_readability_features(text)
    prediction = compute_flesch_kincaid(readability_features)[-1]

    gold.append(label)
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')
  generalized_results['rule_based']['readability']['flesch_kincaid'][key] = classification_report(gold, predictions, output_dict=True, zero_division=1)

Type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493


KeyboardInterrupt: 

#####**Gunning Fog**

######**Main results**

In [ ]:
for key in texts['main_dataset']:
  main_results['rule_based']['readability']['gunning_fog'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Type: {key}')
  for text in texts['main_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1

    try:
      label, text = text.split(" ",maxsplit=1)
    except ValueError:
      counter = counter - 1
      continue

    readability_features = extract_readability_features(text)
    prediction = compute_gunning_fog(readability_features)[-1]

    gold.append(label)
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')
  main_results['rule_based']['readability']['gunning_fog'][key] = classification_report(gold, predictions, output_dict=True, zero_division=1)

Type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 1.lowercased
Predicting text 500 out of 1493


KeyboardInterrupt: 

######**Generalized results**

In [ ]:
for key in texts['generalized_dataset']:
  generalized_results['rule_based']['readability']['gunning_fog'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Type: {key}')
  for text in texts['generalized_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1

    try:
      label, text = text.split(" ",maxsplit=1)
    except ValueError:
      counter = counter - 1
      continue

    readability_features = extract_readability_features(text)
    prediction = compute_gunning_fog(readability_features)[-1]

    gold.append(label)
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')
  generalized_results['rule_based']['readability']['gunning_fog'][key] = classification_report(gold, predictions, output_dict=True, zero_division=1)

Type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493
Type: 1.lowercased
Predicting text 500 out of 1493
Predicting text 1000 out of 1493


KeyboardInterrupt: 

#####**Coleman Liau**

######**Main results**

In [ ]:
for key in texts['main_dataset']:
  main_results['rule_based']['readability']['coleman_liau'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Type: {key}')
  for text in texts['main_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1

    try:
      label, text = text.split(" ",maxsplit=1)
    except ValueError:
      counter = counter - 1
      continue

    readability_features = extract_readability_features(text)
    prediction = compute_coleman_liau(readability_features)[-1]

    gold.append(label)
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')
  main_results['rule_based']['readability']['coleman_liau'][key] = classification_report(gold, predictions, output_dict=True, zero_division=1)

Type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493


KeyboardInterrupt: 

######**Generalized results**

In [ ]:
for key in texts['generalized_dataset']:
  generalized_results['rule_based']['readability']['coleman_liau'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Type: {key}')
  for text in texts['generalized_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1

    try:
      label, text = text.split(" ",maxsplit=1)
    except ValueError:
      counter = counter - 1
      continue

    readability_features = extract_readability_features(text)
    prediction = compute_coleman_liau(readability_features)[-1]

    gold.append(label)
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')
  generalized_results['rule_based']['readability']['coleman_liau'][key] = classification_report(gold, predictions, output_dict=True, zero_division=1)

#####**ARI**

######**Main results**

In [ ]:
for key in texts['main_dataset']:
  main_results['rule_based']['readability']['ARI'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Type: {key}')
  for text in texts['main_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1

    try:
      label, text = text.split(" ",maxsplit=1)
    except ValueError:
      counter = counter - 1
      continue

    readability_features = extract_readability_features(text)
    prediction = compute_ari(readability_features)[-1]

    gold.append(label)
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')
  main_results['rule_based']['readability']['ARI'][key] = classification_report(gold, predictions, output_dict=True, zero_division=1)

Type: 0.raw
Predicting text 500 out of 1493
Predicting text 1000 out of 1493


KeyboardInterrupt: 

######**Generalized results**

In [ ]:
for key in texts['generalized_dataset']:
  generalized_results['rule_based']['readability']['ARI'][key] = {}

  gold = []
  predictions = []

  counter = 0
  print(f'Type: {key}')
  for text in texts['generalized_dataset'][key]['test'][:index_of_texts_to_predict]:
    counter = counter + 1

    try:
      label, text = text.split(" ",maxsplit=1)
    except ValueError:
      counter = counter - 1
      continue

    readability_features = extract_readability_features(text)
    prediction = compute_ari(readability_features)[-1]

    gold.append(label)
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict}')
  generalized_results['rule_based']['readability']['ARI'][key] = classification_report(gold, predictions, output_dict=True, zero_division=1)

Type: 0.raw
Predicting text 500 out of 1493


KeyboardInterrupt: 

####**Machine Learning**

#####**Metodi classici**

######**Random Forest**

**Main results**

Servono circa 12 minuti

In [ ]:
for key in texts['main_dataset']:
  main_results['machine_learning']['traditional']['random_forest'][key] = {}
  gold = []
  predictions = []
  model = random_forest_models[key]
  counter = 0

  for label, text in zip(vectorized_texts['main_dataset'][key]['test']['labels'][:index_of_texts_to_predict], vectorized_texts['main_dataset'][key]['test']['texts_as_vectors'][:index_of_texts_to_predict]):
    counter = counter + 1

    prediction = traditional_model_predict(model, text)
    gold.append(int_to_cefr[label])
    predictions.append(prediction)

    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict} --- Split: {key}')

  print(key)
  print(classification_report(gold, predictions))
  main_results['machine_learning']['traditional']['random_forest'][key] = classification_report(gold, predictions, output_dict=True)

KeyError: '0.raw'

**Generalized results**

Servono circa 12 minuti

In [ ]:
for key in texts['generalized_dataset']:
  generalized_results['machine_learning']['traditional']['random_forest'][key] = {}
  gold = []
  predictions = []
  model = random_forest_models[key]
  counter = 0

  for label, text in zip(vectorized_texts['generalized_dataset'][key]['test']['labels'][:index_of_texts_to_predict], vectorized_texts['generalized_dataset'][key]['test']['texts_as_vectors'][:index_of_texts_to_predict]):
    counter = counter + 1
    try:
      prediction = traditional_model_predict(model, text)
      gold.append(int_to_cefr[label])
      predictions.append(prediction)
    except ValueError:
      counter = counter - 1
      continue

    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict} --- Split: {key}')

  print(key)
  print(classification_report(gold, predictions))
  generalized_results['machine_learning']['traditional']['random_forest'][key] = classification_report(gold, predictions, output_dict=True)

Predicting text 100 out of 1493 --- Split: 0.raw
Predicting text 200 out of 1493 --- Split: 0.raw
Predicting text 300 out of 1493 --- Split: 0.raw
Predicting text 400 out of 1493 --- Split: 0.raw
Predicting text 500 out of 1493 --- Split: 0.raw
Predicting text 600 out of 1493 --- Split: 0.raw
Predicting text 700 out of 1493 --- Split: 0.raw
Predicting text 800 out of 1493 --- Split: 0.raw
Predicting text 900 out of 1493 --- Split: 0.raw
Predicting text 1000 out of 1493 --- Split: 0.raw
Predicting text 1100 out of 1493 --- Split: 0.raw
Predicting text 1200 out of 1493 --- Split: 0.raw
Predicting text 1300 out of 1493 --- Split: 0.raw
Predicting text 1400 out of 1493 --- Split: 0.raw
0.raw
              precision    recall  f1-score   support

 __label__A1       0.08      0.84      0.15        82
 __label__A2       0.16      0.08      0.11       327
 __label__B1       0.30      0.04      0.07       391
 __label__B2       0.32      0.12      0.17       472
 __label__C1       0.22      0.2

######**Logistic Regression**

**Main results**

In [ ]:
for key in texts['main_dataset']:
  main_results['machine_learning']['traditional']['logistic_regression'][key] = {}
  gold = []
  predictions = []
  model = logistic_regression_models[key]
  counter = 0

  for label, text in zip(vectorized_texts['main_dataset'][key]['test']['labels'][:index_of_texts_to_predict], vectorized_texts['main_dataset'][key]['test']['texts_as_vectors'][:index_of_texts_to_predict]):
    counter = counter + 1
    print(f'Predicting text {counter} out of {index_of_texts_to_predict} --- Split: {key}')

    prediction = traditional_model_predict(model, text)
    gold.append(int_to_cefr[label])
    predictions.append(prediction)

    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict} --- Split: {key}')

  print(key)
  print(classification_report(gold, predictions))
  main_results['machine_learning']['traditional']['logistic_regression'][key] = classification_report(gold, predictions, output_dict=True)

Streaming output truncated to the last 5000 lines.
Predicting text 1031 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1032 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1033 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1034 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1035 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1036 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1037 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1038 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1039 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1040 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1041 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1042 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1043 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1044 out o

**Generalized results**

In [ ]:
for key in texts['generalized_dataset']:
  generalized_results['machine_learning']['traditional']['logistic_regression'][key] = {}
  gold = []
  predictions = []
  model = logistic_regression_models[key]
  counter = 0

  for label, text in zip(vectorized_texts['generalized_dataset'][key]['test']['labels'][:index_of_texts_to_predict], vectorized_texts['generalized_dataset'][key]['test']['texts_as_vectors'][:index_of_texts_to_predict]):
    counter = counter + 1
    print(f'Predicting text {counter} out of {index_of_texts_to_predict} --- Split: {key}')
    try:
      prediction = traditional_model_predict(model, text)
      gold.append(int_to_cefr[label])
      predictions.append(prediction)
    except ValueError:
      counter = counter -1
      continue
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict} --- Split: {key}')

  print(key)
  print(classification_report(gold, predictions))
  generalized_results['machine_learning']['traditional']['logistic_regression'][key] = classification_report(gold, predictions, output_dict=True)

Streaming output truncated to the last 5000 lines.
Predicting text 1031 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1032 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1033 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1034 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1035 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1036 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1037 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1038 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1039 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1040 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1041 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1042 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1043 out of 1493 --- Split: 5.lowercased_nopunctuation
Predicting text 1044 out o

######**Gradient Boosting**

**Main results**

In [ ]:
for key in texts['main_dataset']:
  main_results['machine_learning']['traditional']['gradient_boosting'][key] = {}
  gold = []
  predictions = []
  model = gradient_boosting_models[key]
  counter = 0

  for label, text in zip(vectorized_texts['main_dataset'][key]['test']['labels'][:index_of_texts_to_predict], vectorized_texts['main_dataset'][key]['test']['texts_as_vectors'][:index_of_texts_to_predict]):
    counter = counter + 1

    prediction = traditional_model_predict(model, text)
    gold.append(int_to_cefr[label])
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict} --- Split: {key}')
  print(key)
  print(classification_report(gold, predictions))
  main_results['machine_learning']['traditional']['gradient_boosting'][key] = classification_report(gold, predictions, output_dict=True)

Predicting text 100 out of 1493 --- Split: 0.raw
Predicting text 200 out of 1493 --- Split: 0.raw
Predicting text 300 out of 1493 --- Split: 0.raw
Predicting text 400 out of 1493 --- Split: 0.raw
Predicting text 500 out of 1493 --- Split: 0.raw
Predicting text 600 out of 1493 --- Split: 0.raw
Predicting text 700 out of 1493 --- Split: 0.raw
Predicting text 800 out of 1493 --- Split: 0.raw
Predicting text 900 out of 1493 --- Split: 0.raw
Predicting text 1000 out of 1493 --- Split: 0.raw
Predicting text 1100 out of 1493 --- Split: 0.raw
Predicting text 1200 out of 1493 --- Split: 0.raw
Predicting text 1300 out of 1493 --- Split: 0.raw
Predicting text 1400 out of 1493 --- Split: 0.raw
0.raw
              precision    recall  f1-score   support

 __label__A1       0.85      0.85      0.85       282
 __label__A2       0.67      0.65      0.66       295
 __label__B1       0.64      0.69      0.66       298
 __label__B2       0.62      0.57      0.59       289
 __label__C1       0.81      0.8

**Generalized results**

In [ ]:
for key in texts['generalized_dataset']:
  generalized_results['machine_learning']['traditional']['gradient_boosting'][key] = {}
  gold = []
  predictions = []
  model = gradient_boosting_models[key]
  counter = 0

  for label, text in zip(vectorized_texts['generalized_dataset'][key]['test']['labels'][:index_of_texts_to_predict], vectorized_texts['generalized_dataset'][key]['test']['texts_as_vectors'][:index_of_texts_to_predict]):
    counter = counter + 1

    prediction = traditional_model_predict(model, text)
    gold.append(int_to_cefr[label])
    predictions.append(prediction)
    if counter % 500 == 0:
      print(f'Predicting text {counter} out of {index_of_texts_to_predict} --- Split: {key}')
  print(key)
  print(classification_report(gold, predictions))
  generalized_results['machine_learning']['traditional']['gradient_boosting'][key] = classification_report(gold, predictions, output_dict=True)

Predicting text 100 out of 1493 --- Split: 0.raw
Predicting text 200 out of 1493 --- Split: 0.raw
Predicting text 300 out of 1493 --- Split: 0.raw
Predicting text 400 out of 1493 --- Split: 0.raw
Predicting text 500 out of 1493 --- Split: 0.raw
Predicting text 600 out of 1493 --- Split: 0.raw
Predicting text 700 out of 1493 --- Split: 0.raw
Predicting text 800 out of 1493 --- Split: 0.raw
Predicting text 900 out of 1493 --- Split: 0.raw
Predicting text 1000 out of 1493 --- Split: 0.raw
Predicting text 1100 out of 1493 --- Split: 0.raw
Predicting text 1200 out of 1493 --- Split: 0.raw
Predicting text 1300 out of 1493 --- Split: 0.raw
Predicting text 1400 out of 1493 --- Split: 0.raw
0.raw
              precision    recall  f1-score   support

 __label__A1       0.08      0.83      0.15        82
 __label__A2       0.12      0.06      0.08       327
 __label__B1       0.36      0.05      0.08       391
 __label__B2       0.29      0.10      0.15       472
 __label__C1       0.26      0.3

#####**FastText**


######**Main results**

In [ ]:
for key in texts['main_dataset']:
  main_results['machine_learning']['fasttext'][key] = {}
  gold = []
  predictions = []
  model_path = f'/content/material_notebook_2/other_stuff/fasttext_models/{key}_fasttext_model.bin'
  model = fasttext.load_model(model_path)
  for text in texts['main_dataset'][key]['test'][:index_of_texts_to_predict]:
    label, text = text.split(" ",maxsplit=1)
    prediction = model.predict([text])[0][0][0]
    gold.append(label)
    predictions.append(prediction)
  print(classification_report(gold, predictions))
  main_results['machine_learning']['fasttext'][key] = classification_report(gold, predictions, output_dict=True)

              precision    recall  f1-score   support

 __label__A1       0.97      0.97      0.97       282
 __label__A2       0.96      0.93      0.94       295
 __label__B1       0.93      0.92      0.93       298
 __label__B2       0.93      0.92      0.93       289
 __label__C1       0.94      0.99      0.97       329

    accuracy                           0.95      1493
   macro avg       0.95      0.95      0.95      1493
weighted avg       0.95      0.95      0.95      1493

              precision    recall  f1-score   support

 __label__A1       0.96      0.95      0.96       282
 __label__A2       0.94      0.95      0.95       295
 __label__B1       0.93      0.91      0.92       298
 __label__B2       0.94      0.93      0.94       289
 __label__C1       0.94      0.97      0.96       329

    accuracy                           0.94      1493
   macro avg       0.94      0.94      0.94      1493
weighted avg       0.94      0.94      0.94      1493

              precisio

######**Generalized results**

In [ ]:
for key in texts['generalized_dataset']:

  generalized_results['machine_learning']['fasttext'][key] = {}

  model_path = f'/content/material_notebook_2/other_stuff/fasttext_models/{key}_fasttext_model.bin'

  model = fasttext.load_model(model_path)

  gold = []
  predictions = []
  for text in texts['generalized_dataset'][key]['test']:
    try:
      label, text = text.split(" ",maxsplit=1)
      prediction = model.predict([text])[0][0][0]
      gold.append(label)
      predictions.append(prediction)
    except ValueError:
      continue
  generalized_results['machine_learning']['fasttext'][key] = classification_report(gold, predictions, output_dict=True)

####**LLM**
Tramite questi due cicli è possibile predire le etichette dei vari testi:

In [ ]:
os.makedirs(r'gemini_results')

#####**Predizioni *Few-Shot***

In [ ]:
gold = []
predictions = []
counter = 0
for text in texts['main_dataset']['0.raw']['test'][:index_of_texts_to_predict]:
    counter = counter + 1
    print(f'Text {counter} out of 1500')
    label, text = text.split(" ",maxsplit=1)
    prediction = gemini_predictor(prompt_few_shot, text)
    print(label, prediction)
    gold.append(label)
    predictions.append(prediction)

with open(r'/content/gemini_results/gold_predictions_few_shot.json','w',encoding='utf-8') as file:
  content = {'gold':gold, 'predictions':predictions}
  json.dump(content,file,indent=4)

main_results['llm']['gemini']['few_shot']['0.raw'] = classification_report(gold, predictions, output_dict=True,zero_division=1)

#####**Predizioni *Zero-Shot***

In [ ]:
gold = []
predictions = []
counter = 0
for text in texts['main_dataset']['0.raw']['test'][:index_of_texts_to_predict]:
    counter = counter + 1
    print(f'Text {counter} out of 1500')
    label, text = text.split(" ",maxsplit=1)
    prediction = gemini_predictor(prompt_zero_shot, text)
    print(label, prediction)
    gold.append(label)
    predictions.append(prediction)
with open(r'/content/gemini_results/gold_predictions_zero_shot.json','w',encoding='utf-8') as file:
  content = {'gold':gold, 'predictions':predictions}
  json.dump(content,file,indent=4)

main_results['llm']['gemini']['zero_shot']['0.raw'] = classification_report(gold, predictions, output_dict=True,zero_division=1)

Dato che, con le poche richieste possibili al giorno, non è possibile eseguirne 3000 in una sessione, nella cella sottostante si possono scaricare le predizioni fatte in precedenza.

Dopo averle scaricate, si inseriscono nel dizionario *main_results*:

In [ ]:
main_results['LLM']['gemini']['few_shot']['0.raw'] = {}

with open(r'/content/material_notebook_2/other_stuff/gemini_results/gold_predictions_few_shot.json','r',encoding='utf-8') as file:
  content = json.load(file)
  gold = content['gold']
  predictions = content['predictions']
  main_results['LLM']['gemini']['few_shot']['0.raw'] = classification_report(gold, predictions, output_dict=True,zero_division=1)

In [ ]:
main_results['LLM']['gemini']['zero_shot']['0.raw'] = {}

with open(r'/content/material_notebook_2/other_stuff/gemini_results/gold_predictions_zero_shot.json','r',encoding='utf-8') as file:
  content = json.load(file)
  gold = content['gold']
  predictions = content['predictions']
  main_results['LLM']['gemini']['zero_shot']['0.raw'] = classification_report(gold, predictions, output_dict=True,zero_division=1)

###**Salvataggio su Json**
Infine, salviamo i risultati sui due file *json* e zippiamoli:

In [ ]:
with open('/content/gemini_results/main_results.json', 'w', encoding='utf-8') as f:
    json.dump(main_results,f,indent=4)

In [ ]:
with open('/content/gemini_results/generalized_results.json', 'w', encoding='utf-8') as f:
    json.dump(generalized_results,f,indent=4)

In [ ]:
folder_to_zip = '/content/gemini_results'
output_filename = 'material_notebook_3'
shutil.make_archive(output_filename, 'zip', folder_to_zip)

'/content/material_notebook_3.zip'

#**Altri notebook**
Numero|Titolo|URL
----|------|-------
1 / 3|Introduzione, gestione dei dataset e preprocessing|[link](https://colab.research.google.com/drive/11klGGwytqSNmE_WaWieAbVbtZL_qiWkl?authuser=4#scrollTo=75d3EdOJariY)
**2 / 3**|**Costruzione dei modelli, training e primo testing**|[link](https://colab.research.google.com/drive/1XXkGnLBFJmKjBbNl1a-1Q-73PIfT2QpZ?authuser=4#scrollTo=2PgXRu6P2iTy)
3 / 3|Grafici e conclusioni|[link](https://colab.research.google.com/drive/1-jy9wDUCTJ5TB1xcsnP9B_6wdTFvktw6?authuser=4#scrollTo=TT9PH3RlpgML)